In [9]:
## Execution timing
import time
from datetime import datetime
startTime = time.time()

In [10]:
## Load h2o and prerequisites libraries
import h2o
import numpy as np
import pandas as pd
from h2o.estimators.gbm import H2OGradientBoostingEstimator 
from h2o.grid.grid_search import H2OGridSearch
import warnings
import json
import pyodbc # import oracledb

import config

h2o.init(nthreads=-1, min_mem_size= config.min_memory, max_mem_size= config.max_memory, enable_assertions=False)
warnings.filterwarnings('ignore')

# oracledb.init_oracle_client(lib_dir=r"E:/Oracle/product/instantclient_12_1")

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,5 mins 50 secs
H2O_cluster_timezone:,America/New_York
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.44.0.2
H2O_cluster_version_age:,3 months and 7 days
H2O_cluster_name:,H2O_from_python_randle_knight_7ot70e
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,28.30 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"


In [11]:
# Check lims dsn connection to database before proceeding
def get_lims_connection():
    global odbcLimsConnection
    # odbcLimsConnection = oracledb.connect(user=config.limsdb_username, password=config.limsdb_password,
    # dsn=config.limsdb_dsn)
    odbcLimsConnection = pyodbc.connect('DRIVER=' + config.limsdb_driver + ';DATABASE=' + config.limsdb_database +
                                        ';UID=' + config.limsdb_username + ';PWD=' + config.limsdb_password +
                                        ';SERVER=' + config.limsdb_hostname + ';PORT=' + config.limsdb_port +
                                        ';Trusted_Connection=yes')
    return odbcLimsConnection


if __name__ == '__main__':
    try:  # GET THE CONNECTION OBJECT (ENGINE) FOR THE DATABASE
        get_lims_connection()
    except Exception as error:
        exit("Connection could not be made due to the following error: \n", error)


# Check mlar dsn connection to database before proceeding
def get_mlar_connection():
    global odbcMlarConnection
    # odbcMlarConnection = oracledb.connect(user=config.mlardb_username, password=config.mlardb_password,
    # dsn=config.mlardb_dsn)
    odbcMlarConnection = pyodbc.connect('DRIVER=' + config.mlardb_driver + ';DATABASE=' + config.mlardb_database +
                                        ';UID=' + config.mlardb_username + ';PWD=' + config.mlardb_password +
                                        ';SERVER=' + config.mlardb_hostname + ';PORT=' + config.mlardb_port +
                                        ';Trusted_Connection=yes')
    return odbcMlarConnection


if __name__ == '__main__':
    try:
        # GET THE CONNECTION OBJECT (ENGINE) FOR THE DATABASE
        get_mlar_connection()
    except Exception as error:
        exit("Connection could not be made due to the following error: \n", error)

In [12]:
# Clear h2o memory
h2o.remove_all()

# Model location
model_path = config.model_path

In [13]:
# Query labelled data from isolation forest.
QueryMode = True
if QueryMode == True:
    # Start and Finish Logs - MOVE TO CONFIG
    #sql_command = "select MIN(AUDIT_TIMESTAMP), DateAdd(MIN(AUDIT_TIMESTAMP), , , , 1) from SAMPLE_AUDIT_DECRYPT" # Oracle handling
    sql_command = "select MIN(AUDIT_TIMESTAMP),MIN(AUDIT_TIMESTAMP) + INTERVAL '1 year' from SAMPLE_AUDIT_DECRYPT"
    with odbcLimsConnection.cursor() as lims_cursor_fetch_one:
        lims_cursor_fetch_one.execute(sql_command)
        response = lims_cursor_fetch_one.fetchone()
    startLog = response[0]
    endLog = response[1]
    
    cursor = odbcMlarConnection.cursor()
    column_names = ['mlar_type', 'audit_action', 'audit_timestamp', 'audit_type', 'audit_counter', 'trans_order',
                'table_name', 'table_key', 'reason', 'record_signed', 'user_name', 'user_role', 'parent_table',
                'parent_key', 'batch_name', 'col_num', 'document_number', 'experiment_number', 'row_num',
                'workbook_number', 'worksheet', 'formulation', 'investigation', 'order_num', 'project_name', 'name',
                'report_number', 'sample_number', 'location_number', 't_mlar', 't_mlar_severity', 'trans_string']
    sql_command = "select 'sample_audit_log',action,audit_timestamp,audit_type,counter,trans_order,table_name,table_key," \
              "reason,record_signed,user_name,user_role,sample_number,t_mlar,t_mlar_severity,trans_string" \ 
              " from SAMPLE_AUDIT_DECRYPT where audit_timestamp " \
              "between '" + str(startLog) + "' and '" + str(endLog) + "' and t_mlar = 'T'"
    with odbcMlarConnection.cursor() as mlar_cursor_fetch_all:
        mlar_cursor_fetch_all.execute(sql_command)
        queryFetch = mlar_cursor_fetch_all.fetchall()
    
    dataFrame = pd.DataFrame.from_records(queryFetch, columns=column_names)

    # Retrieve datatypes for  Dataframe
    # "string" | "float" | "real" | "double" | "int" | "long" | "numeric" | "categorical" | "factor" | "enum" | "time"
    sql_command = "select distinct FIELD_NAME, CASE WHEN FIELD_NAME = 'T_MLAR_SEVERITY' THEN 'categorical' WHEN DATA_TYPE " \
              "in ('Text','Long','List','MultiList','Boolean','File','DBFile','Interval','DateTime','Date') THEN " \
              "'string' WHEN DATA_TYPE in ('Integer','Number') THEN 'numeric' ELSE DATA_TYPE END AS DATATYPE from " \
              "FIELD_MASTER where TABLE_NAME in ('SAMPLE_AUDIT_LOG','SAMPLE','TEST','RESULT') order by FIELD_NAME"

    with odbcLimsConnection.cursor() as lims_cursor_fetch_all:
        lims_cursor_fetch_all.execute(sql_command)
        queryFetch = lims_cursor_fetch_all.fetchall()

    fieldDict = queryFetch

    # Convert JSON field to JSON
    # dataFrame['trans_string'].to_json(path_or_buf="audit.json", orient="records")
    dataFrame['trans_string'].apply(json.loads).to_json(path_or_buf="audit.json", orient="records")
    with open("audit.json") as jsonEntries:
        jsonDataFrame = json.load(jsonEntries)
    jsonDataFrame = pd.json_normalize(jsonDataFrame)

    # Convert column datatypes to dictionary
    fieldDictDF = pd.DataFrame.from_records(fieldDict)
    fieldDictDF.set_index(0, inplace=True)
    fieldDict = fieldDictDF[1].to_dict()

    # Drop unnecessary or duplicated columns, replace NaN values and comma-replacement character
    dataFrame = dataFrame.drop(columns=['trans_string'])
    # jsonDataFrame = jsonDataFrame.drop(columns=['SAMPLE_NUMBER']).replace(np.nan,-999).replace("▄",",",regex=True)
    jsonDataFrame = jsonDataFrame.replace(np.nan, "█").replace("▄", ",", regex=True)
    encodedDataFrame = pd.concat([dataFrame, jsonDataFrame], axis=1)
    unsupervisedFeatures = list(encodedDataFrame.drop(columns=['t_mlar']).columns.values)
    
    # Define datatypes and convert to H2oFrame
    focus_column = "t_mlar_severity"
    
    supervisedFeatures = list(encodedDataFrame.columns.values)
    fieldDictSupervised = dict(filter(lambda item: item[0] in supervisedFeatures, fieldDict.items()))
        
    supervisedH2oFrame = h2o.H2OFrame(python_obj=encodedDataFrame, column_types=fieldDictSupervised)
    supervisedH2oFrame["t_mlar_severity"] = supervisedH2oFrame["t_mlar_severity"].asfactor()
    supervisedH2oFrameTrain, supervisedH2oFrameTest = supervisedH2oFrame.split_frame(ratios=[.8], seed=1234)
else:
    supervisedH2oFrame = h2o.import_file("")
    supervisedH2oFrameTrain, supervisedH2oFrameTest = supervisedH2oFrame.split_frame(ratios=[.8], seed=1234)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [20]:
# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit, Data load Time = " + str(elapsedTime)
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

DateTime: 2024-02-16 02:49:19.807963, Start: 2023-09-06 18:20:12, End: 2024-09-06 18:20:12, Sample Audit, Data load Time = 176.07719922065735


In [22]:
trainMode = True
if trainMode == True:
    gbm_params1 = {'learn_rate': [0.01, 0.1, 0.4, 0.7, 1.0],
                'max_depth': [12, 14, 16, 18, 20],
                'sample_rate': [0.1, 0.3, 0.7, 1.0],
                'col_sample_rate': [0.2, 0.5, 1.0, 1.5, 2.0]}
    gbm_grid1 = H2OGridSearch(model=H2OGradientBoostingEstimator, grid_id='gbm_grid1', hyper_params=gbm_params1)
    gbm_grid1.train(x=supervisedFeatures, y=focus_column, training_frame=supervisedH2oFrameTrain, validation_frame=supervisedH2oFrameTest, ntrees=100, seed=1)
    h2o.download_model(model=supervisedModel, path="Download")
else:
    supervisedModel = h2o.load_model(model_path)

# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit, Load Model Time = " + str(elapsedTime)
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

gbm Model Build progress: |██████████████████████████████████████████████████████| (done) 100%
DateTime: 2024-02-16 02:52:57.733249, Start: 2023-09-06 18:20:12, End: 2024-09-06 18:20:12, Sample Audit, Load Model Time = 394.00240778923035


In [23]:
performance = supervisedModel.model_performance(supervisedH2oFrameTest)
print(performance)

# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit, Model Performance Time = " + str(elapsedTime)
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

ModelMetricsMultinomial: gbm
** Reported on test data. **

MSE: 0.005723695715342087
RMSE: 0.07565511030553115
LogLoss: 0.017722831771764917
Mean Per-Class Error: 0.026843229808236307
AUC table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).
AUCPR table was not computed: it is either disabled (model parameter 'auc_type' was set to AUTO or NONE) or the domain size exceeds the limit (maximum is 50 domains).

Confusion Matrix: Row labels: Actual class; Column labels: Predicted class
0       1      4    Error       Rate
------  -----  ---  ----------  ---------------
165579  943    7    0.00570471  950 / 166,529
521     9088   2    0.0544168   523 / 9,611
1       0      48   0.0204082   1 / 49
166101  10031  57   0.00836602  1,474 / 176,189

Top-3 Hit Ratios: 
k    hit_ratio
---  -----------
1    0.991634
2    0.999989
3    1
DateTime: 2024-02-16 02:53:48.775568, Start: 2023-09-06 18:

In [24]:
predictSupervised = supervisedModel.predict(supervisedH2oFrameTest)
predictSupervised = predictSupervised.cbind(supervisedH2oFrameTest[unsupervisedFeatures])

# Execution timing
endTime = time.time()
elapsedTime = endTime - startTime
now = datetime.now()
logEntry = "DateTime: " + str(now) + ", Start: " + str(startLog) + ", End: " + str(endLog) + ", Sample Audit, Prediction Time = " + str(elapsedTime)
with open ("executionlog.txt", "a") as writer:
    writer.write('\n'+logEntry)
    writer.close()
    print(logEntry)

gbm prediction progress: |███████████████████████████████████████████████████████| (done) 100%
DateTime: 2024-02-16 02:53:59.678310, Start: 2023-09-06 18:20:12, End: 2024-09-06 18:20:12, Sample Audit, Prediction Time = 455.947541475296


In [25]:
#predictSupervised

In [26]:
#predictSupervised["t_mlar_severity"].asfactor().grep("1", output_logical=True)

In [27]:
mask = predictSupervised["t_mlar_severity"] > 0
cols = predictSupervised[mask,:]

In [28]:
#cols
#confusionMatrix = supervisedModel.hit_ratio_table(valid=True)

In [30]:
## Clear h2o memory
h2o.remove_all()
h2o.cluster().shutdown()

H2O session _sid_9ef2 closed.


In [14]:
str(startLog)  

'2022-08-18 18:35:18'

In [15]:
str(endLog)

'2023-08-18 18:35:18'